# 00 — Project Setup

This notebook prepares the local environment for the Time Series PILE forecasting benchmark.

It performs the following tasks:

- Adds the project root to the Python path.
- Downloads the Time Series PILE forecasting subset from Hugging Face if needed.
- Locates the Monash `.tsf` forecasting dataset folder.
- Sets the `MONASH_TSF_BASE_DIR` environment variable.
- Verifies that the shared dataset loader can find and load selected datasets.

This notebook should be run before executing the dataset experiment notebooks.

## 1. Add project root to Python path

The source code is stored under the `src/` directory. This step adds the project root to the Python path so that modules such as `src.data.dataset_loader` can be imported from notebooks.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)

Project root added: /Users/kunalgurung/Desktop/UTS SEM4/36127/timeseries-pile-foundation-model-benchmark


## 2. Import setup dependencies

The setup notebook uses `pathlib` for local path handling and `huggingface_hub` to download the Time Series PILE forecasting files.

In [2]:
import os
from pathlib import Path

from huggingface_hub import snapshot_download

/Users/kunalgurung/.pyenv/versions/3.11.4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Download the forecasting subset

The project uses the forecasting subset of the Time Series PILE dataset. The download is restricted to the `forecasting/` folder to avoid downloading unnecessary files.

In [3]:
repo_id = "AutonLab/Timeseries-PILE"

local_snapshot_path = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["forecasting/**"],
    max_workers=2,
)

print("Downloaded snapshot path:")
print(local_snapshot_path)

Fetching ... files: 58it [00:00, 6852.67it/s]

Downloaded snapshot path:
/Users/kunalgurung/.cache/huggingface/hub/datasets--AutonLab--Timeseries-PILE/snapshots/ea89753da2b451928436adb333c7a2e892461c7d


## 4. Locate the Monash `.tsf` folder

The selected datasets are stored in Monash `.tsf` format under the forecasting subset.

In [4]:
monash_base_dir = Path(local_snapshot_path) / "forecasting" / "monash"

print("Expected Monash folder:")
print(monash_base_dir)
print("Exists:", monash_base_dir.exists())

Expected Monash folder:
/Users/kunalgurung/.cache/huggingface/hub/datasets--AutonLab--Timeseries-PILE/snapshots/ea89753da2b451928436adb333c7a2e892461c7d/forecasting/monash
Exists: True


## 5. Set the dataset directory environment variable

The dataset loader reads the `MONASH_TSF_BASE_DIR` environment variable to locate `.tsf` files.

In [5]:
os.environ["MONASH_TSF_BASE_DIR"] = str(monash_base_dir)

print("MONASH_TSF_BASE_DIR set to:")
print(os.environ["MONASH_TSF_BASE_DIR"])

MONASH_TSF_BASE_DIR set to:
/Users/kunalgurung/.cache/huggingface/hub/datasets--AutonLab--Timeseries-PILE/snapshots/ea89753da2b451928436adb333c7a2e892461c7d/forecasting/monash


## 6. Verify available `.tsf` files

This step checks whether the Monash forecasting directory contains `.tsf` files.

In [6]:
tsf_files = sorted(monash_base_dir.glob("*.tsf"))

print("Number of .tsf files found:", len(tsf_files))
print("\nFirst 10 files:")
for file_path in tsf_files[:10]:
    print("-", file_path.name)

Number of .tsf files found: 47

First 10 files:
- australian_electricity_demand_dataset.tsf
- bitcoin_dataset_without_missing_values.tsf
- car_parts_dataset_without_missing_values.tsf
- cif_2016_dataset.tsf
- covid_deaths_dataset.tsf
- dominick_dataset.tsf
- electricity_hourly_dataset.tsf
- electricity_weekly_dataset.tsf
- fred_md_dataset.tsf
- hospital_dataset.tsf


## 7. Test shared dataset loader imports

The project uses reusable data loading functions from `src.data.dataset_loader`.

In [7]:
from src.data.dataset_loader import list_available_datasets, load_monash_dataset

## 8. List available datasets

This step confirms which datasets are available after downloading the Time Series PILE forecasting subset.

In [8]:
available_datasets = list_available_datasets()
print("Number of available datasets:", len(available_datasets))
print("\nFirst 20 dataset keys:")
print(available_datasets[:20])

Number of available datasets: 47

First 20 dataset keys:
['australian_electricity_demand', 'bitcoin', 'car_parts', 'cif_2016', 'covid_deaths', 'dominick', 'electricity_hourly', 'electricity_weekly', 'fred_md', 'hospital', 'kaggle_web_traffic', 'kaggle_web_traffic_weekly', 'kdd_cup_2018', 'london_smart_meters', 'm1_monthly', 'm1_quarterly', 'm1_yearly', 'm3_monthly', 'm3_other', 'm3_quarterly']


## 9. Test loading one selected dataset

The `bitcoin` dataset is loaded as a quick sanity check. The output confirms that metadata, horizon information, and split data are being resolved correctly.

In [9]:
bundle = load_monash_dataset("bitcoin")

print("Dataset key:", bundle["dataset_key"])
print("Domain:", bundle["domain"])
print("File name:", bundle["file_name"])
print("Frequency:", bundle["normalized_frequency"])
print("Resolved horizon:", bundle["resolved_horizon"])
print("Horizon strategy:", bundle["horizon_strategy"])
print("Records shape:", bundle["records_df"].shape)
print("Long shape:", bundle["long_df"].shape)
print("Split shape:", bundle["split_df"].shape)

Dataset key: bitcoin
Domain: finance
File name: bitcoin.tsf
Frequency: daily
Resolved horizon: 30
Horizon strategy: fallback_capped
Records shape: (18, 4)
Long shape: (75364, 4)
Split shape: (18, 5)


## 10. Verify selected project datasets

The final benchmark uses nine selected datasets across finance, energy, and retail. This step checks whether all selected datasets are available locally.

In [10]:
from src.data.project_config import SELECTED_DATASETS

print("Selected project datasets:")
for dataset_key in SELECTED_DATASETS:
    print("-", dataset_key, "| available =", dataset_key in available_datasets)

Selected project datasets:
- bitcoin | available = True
- fred_md | available = True
- m1_monthly | available = True
- solar_weekly | available = True
- saugeenday | available = True
- electricity_weekly | available = True
- car_parts | available = True
- nn5_weekly | available = True
- nn5_daily | available = True


## 11. Setup completion check

If this cell runs successfully, the project environment is ready for the experiment notebooks.

In [11]:
print("Setup complete.")
print("You can now run project notebooks and experiment scripts.")

Setup complete.
You can now run project notebooks and experiment scripts.


## What to do next

After this notebook runs successfully, execute the dataset experiment notebooks in order:

1. `03_electricity_weekly_experiment.ipynb`
2. `04_saugeenday_experiment.ipynb`
3. `05_solar_weekly_experiment.ipynb`
4. `06_bitcoin_experiment.ipynb`
5. `07_fred_md_experiment.ipynb`
6. `08_m1_monthly_experiment.ipynb`
7. `09_nn5_daily_experiment.ipynb`
8. `10_ttm_adaptation_nn5_daily.ipynb`
9. `11_nn5_weekly_experiment.ipynb`
10. `12_ttm_adaptation_nn5_weekly.ipynb`
11. `13_car_parts_experiment.ipynb`
12. `14_final_analysis.ipynb`

The final analysis notebook aggregates the saved experiment outputs and produces the final summary tables and visualisations.